# GPU & Performance: Scaling for Speed

Reach for this when you need: 
- Reference for PyTorch 2.x performance features (`torch.compile`).
- Debugging OOM (Out-of-Memory) errors.
- Implementing Automatic Mixed Precision (AMP).

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.checkpoint import checkpoint

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 1. Device Management

| Method | Description | Usage |
| :--- | :--- | :--- |
| `.to(device)` | Move tensor/model to target device | Standard way to handle CPU/GPU swaps |
| `cuda.memory_allocated()` | Bytes currently used by tensors | Debugging memory spikes |
| `cuda.empty_cache()` | Release unused cached memory | Does NOT free memory; just releases room to the OS |

In [ ]:
model = nn.Linear(1000, 1000).to(device)
x = torch.randn(64, 1000).to(device)

if torch.cuda.is_available():
    print(f"Reserved: {torch.cuda.memory_reserved() / 1e6:.2f} MB")
    print(f"Allocated: {torch.cuda.memory_allocated() / 1e6:.2f} MB")

## 2. torch.compile (PyTorch 2.0+)

**torch.compile(model)**
Just-in-Time (JIT) compilation using Triton to fuse kernels and reduce CPU overhead.

✅ **Use when**: GPU training where kernels are small and overhead is high (e.g., Transformers).
❌ **Don't use when**: Rapidly debugging code (it adds startup time) or when using complex control flows inside modules.

In [ ]:
optimized_model = torch.compile(model)

# First pass will be slow (compilation stage); subsequent passes will be faster
out = optimized_model(x)

## 3. Mixed Precision Training (AMP)

**torch.autocast**
Automatically casts tensors to half-precision (`float16` or `bfloat16`) where safe to speed up math.

✅ **Use when**: Any modern GPU (NVIDIA Volta, Ampere+). Essential for throughput.
❌ **Don't use when**: Training requires extremely high precision (e.g. some scientific simulations).

In [ ]:
scaler = torch.cuda.amp.GradScaler() # Handles gradient scaling for float16

for inputs, labels in []:
    with torch.autocast(device_type='cuda', dtype=torch.float16):
        outputs = model(inputs)
        loss = nn.CrossEntropyLoss()(outputs, labels)

    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

## 4. Gradient Checkpointing

**torch.utils.checkpoint**
Trading compute for memory. Recomputes activations during backward pass instead of storing them.

✅ **Use when**: You hit OOM on very large models (e.g., LLMs) even with small batch sizes.
❌ **Don't use when**: Training is already slow; this adds ~30% compute overhead.

In [ ]:
# Wrap bulky sub-modules in checkpoint
# out = checkpoint(bulky_module, inputs)

### Common GPU OOM Errors
- **RuntimeError: CUDA out of memory**: 
  1. Reduce `batch_size`.
  2. Apply `torch.autocast`.
  3. Use `model.zero_grad(set_to_none=True)` to free memory faster than setting to zero.
  4. Use `checkpoint` for deep networks.
- **CPU-GPU Transfer Delay**: Use `pin_memory=True` in DataLoaders.

### Key Takeaways
- `torch.compile` is the single biggest performance lever in PyTorch 2.x.
- Mixed precision (`autocast`) typically gives 2x–4x throughput boost on modern hardware.
- Memory leakage is often caused by stored computation graphs; call `.detach()` if storing metrics.